# Simulated Survey for Similarity Ratings

This notebook simulates a survey with the following design:
- 20 test points (10 test + 10 gen).
- For each test point, 12 training samples:
  - 6 from Data Attribution (DA) rank, one per group.
  - 6 from Similarity rank, one per group.
- Total 240 test–train pairs.
- 20 participants, each does 5 tasks (tests).
- For each assigned test, a participant rates 6 out of its 12 pairs.
- Each pair receives 2–3 ratings (exactly 6 pairs×3 + 6 pairs×2 per test = 30).
- Ratings are integers in [1, 9].

Two rating scenarios are simulated:
1) Fully random ratings.
2) Rank-related + noise (primarily driven by Similarity rank groups).

We produce violin plots for both DA rank groups and Similarity rank groups (6 bins each):
1-10, 10-100, 100-1000, 1000–22000, 22000-27900, 27900-28000.

In [13]:
# Imports and constants
import numpy as np
import pandas as pd
import random
from collections import defaultdict
import plotly.express as px
import plotly.io as pio

np.random.seed(42)
random.seed(42)

# Group labels (6 bins)
GROUP_LABELS = [
    "1-10",
    "10-100",
    "100-1000",
    "1000-22000",
    "22000-27900",
    "27900-28000"
]

# Scenario names
SCENARIO_RANDOM = "Random"
SCENARIO_CORR = "Rank-related + noise"

# Numbers
NUM_TEST = 20  # 10 test + 10 gen
NUM_PARTICIPANTS = 20
TESTS_PER_PARTICIPANT = 5
PAIRS_PER_TEST = 12  # 6 DA + 6 SIM
RATINGS_PER_ASSIGNMENT = 6  # per participant per test

# Rendering (plotly)
try:
    pio.renderers.default = "notebook_connected"
except Exception:
    pass

In [14]:
# Helper functions to generate pairs and schedule

def draw_sim_group_given_da(da_idx: int) -> int:
    """Assign a sim group given DA group with weak correlation (mostly random, sometimes same/neighbor)."""
    r = random.random()
    if r < 0.50:
        return da_idx  # same
    elif r < 0.80:
        neighbor = da_idx + random.choice([-1, 1])
        return max(0, min(5, neighbor))
    else:
        return random.randint(0, 5)  # random

def draw_da_group_given_sim(sim_idx: int) -> int:
    """Assign a DA group given sim group with weak correlation (even weaker)."""
    r = random.random()
    if r < 0.30:
        return sim_idx  # same
    elif r < 0.60:
        neighbor = sim_idx + random.choice([-1, 1])
        return max(0, min(5, neighbor))
    else:
        return random.randint(0, 5)

def generate_pairs_for_test(test_id: str):
    """For a given test, generate 12 pairs: 6 from DA (one per group) and 6 from SIM (one per group).
    Each pair carries both da_group and sim_group indices (0..5) and labels."""
    pairs = []
    # DA-selected
    for gi, glabel in enumerate(GROUP_LABELS):
        sim_idx = draw_sim_group_given_da(gi)
        pair_id = f"{test_id}_DA_{glabel}"
        pairs.append({
            "pair_id": pair_id,
            "test_id": test_id,
            "pair_type": "DA",
            "da_group_idx": gi,
            "da_group": glabel,
            "sim_group_idx": sim_idx,
            "sim_group": GROUP_LABELS[sim_idx]
        })
    # SIM-selected
    for gi, glabel in enumerate(GROUP_LABELS):
        da_idx = draw_da_group_given_sim(gi)
        pair_id = f"{test_id}_SIM_{glabel}"
        pairs.append({
            "pair_id": pair_id,
            "test_id": test_id,
            "pair_type": "SIM",
            "da_group_idx": da_idx,
            "da_group": GROUP_LABELS[da_idx],
            "sim_group_idx": gi,
            "sim_group": glabel
        })
    return pairs

def latin_square_assignments(participants, tests, k_per_participant=5):
    """Assign each participant to k distinct tests; each test gets exactly k participants.
    Uses a simple cyclic Latin-square-like schedule."""
    assert k_per_participant <= len(tests)
    assignments = {p: [] for p in participants}
    for p_idx, p in enumerate(participants):
        for k in range(k_per_participant):
            t = tests[(p_idx + k) % len(tests)]
            assignments[p].append(t)
    counts = defaultdict(int)
    for p, ts in assignments.items():
        for t in ts:
            counts[t] += 1
    for t in tests:
        assert counts[t] == k_per_participant, f"Test {t} count {counts[t]} != {k_per_participant}"
    return assignments

def plan_test_buckets(pair_ids, need_map, num_buckets=5, bucket_size=6, max_tries=1000):
    """Plan 30 rating slots into num_buckets buckets (each size bucket_size).
    Constraint: within a bucket, pair_ids must be unique. need_map gives how many ratings per pair (2 or 3)."""
    assert sum(need_map.values()) == num_buckets * bucket_size == 30
    for _ in range(max_tries):
        buckets = [[] for _ in range(num_buckets)]
        bucket_sets = [set() for _ in range(num_buckets)]
        fills = [0] * num_buckets
        # Place higher-demand pairs first; random tie-break
        order = sorted(pair_ids, key=lambda pid: (-need_map[pid], random.random()))
        ok = True
        for pid in order:
            k = need_map[pid]
            # candidates that don't yet contain pid and have capacity
            cand = [i for i in range(num_buckets) if pid not in bucket_sets[i] and fills[i] < bucket_size]
            if len(cand) < k:
                ok = False
                break
            # choose k least-filled buckets (with random tie-break) for balance
            cand.sort(key=lambda i: (fills[i], random.random()))
            chosen = cand[:k]
            for i in chosen:
                buckets[i].append(pid)
                bucket_sets[i].add(pid)
                fills[i] += 1
        if ok and all(len(b) == bucket_size for b in buckets):
            return buckets
    raise RuntimeError("Failed to plan buckets for test")

def build_rating_tasks(pairs_by_test, participants, tests):
    """Plan rating tasks so each test has 30 ratings.
    Strategy: per-test pre-plan 5 buckets (each 6 unique pairs), then map to assigned participants."""

    # Participant->tests assignments
    assignments = latin_square_assignments(participants, tests, k_per_participant=5)
    # Invert to tests -> list of participants (5 each)
    participants_per_test = {t: [] for t in tests}
    for p, ts in assignments.items():
        for t in ts:
            participants_per_test[t].append(p)
    # Shuffle participant order per test for variability
    for t in tests:
        random.shuffle(participants_per_test[t])

    tasks = []
    for t in tests:
        pair_ids = [p["pair_id"] for p in pairs_by_test[t]]
        # randomly choose 6 pairs to need 3 ratings, rest need 2 (sum=30)
        triplets = set(random.sample(pair_ids, 6))
        need_map = {pid: (3 if pid in triplets else 2) for pid in pair_ids}
        # Plan 5 buckets x 6 items (unique per bucket)
        buckets = plan_test_buckets(pair_ids, need_map, num_buckets=5, bucket_size=6)
        plist = participants_per_test[t]
        assert len(plist) == 5
        for i, pid_list in enumerate(buckets):
            p = plist[i]
            for pid in pid_list:
                tasks.append({"participant_id": p, "test_id": t, "pair_id": pid})

    return tasks

In [15]:
# Generate tests, pairs, tasks, and simulate ratings for both scenarios

# Test ids and types
tests = [f"T{str(i+1).zfill(2)}" for i in range(NUM_TEST)]
test_types = {t: ("test" if i < 10 else "gen") for i, t in enumerate(tests)}

# Participants
participants = [f"P{str(i+1).zfill(2)}" for i in range(NUM_PARTICIPANTS)]

# Pairs per test
pairs_by_test = {t: generate_pairs_for_test(t) for t in tests}
all_pairs_df = pd.DataFrame([p for plist in pairs_by_test.values() for p in plist])

# Build rating tasks schedule
tasks = build_rating_tasks(pairs_by_test, participants, tests)
tasks_df = pd.DataFrame(tasks)

# Join pair attributes into tasks
tasks_df = tasks_df.merge(all_pairs_df, on=["pair_id", "test_id"], how="left")
tasks_df["test_type"] = tasks_df["test_id"].map(test_types)

# Simulate ratings
def simulate_random_rating():
    return int(np.random.randint(1, 10))  # 1..9

def simulate_corr_rating(sim_group_idx: int):
    # Higher score for better similarity group (0 best -> high mean), add noise, on 1..9 scale
    means = [8.2, 7.2, 6.2, 5.0, 4.2, 3.4]  # scaled from 1..5 to 1..9
    mu = means[sim_group_idx]
    noisy = np.clip(np.random.normal(mu, 1.2), 1.0, 9.0)
    return int(np.rint(noisy))  # integer 1..9

ratings_random = tasks_df.copy()
ratings_random["scenario"] = SCENARIO_RANDOM
ratings_random["rating"] = [simulate_random_rating() for _ in range(len(ratings_random))]

ratings_corr = tasks_df.copy()
ratings_corr["scenario"] = SCENARIO_CORR
ratings_corr["rating"] = [simulate_corr_rating(idx) for idx in ratings_corr["sim_group_idx"].tolist()]

# Combine
ratings = pd.concat([ratings_random, ratings_corr], ignore_index=True)

# Basic sanity checks
assert len(tasks_df) == NUM_PARTICIPANTS * TESTS_PER_PARTICIPANT * RATINGS_PER_ASSIGNMENT == 600
counts_per_pair = tasks_df.groupby(["test_id", "pair_id"]).size().rename("n").reset_index()
assert counts_per_pair["n"].between(2, 3).all(), "Each pair must have 2-3 ratings"
print("Total pairs:", all_pairs_df.shape[0], "(expect 240)")
print("Total ratings:", ratings.shape[0], "(expect 1200 for two scenarios)")
ratings.head(3)

Total pairs: 240 (expect 240)
Total ratings: 1200 (expect 1200 for two scenarios)


,participant_id,test_id,pair_id,pair_type,da_group_idx,da_group,sim_group_idx,sim_group,test_type,scenario,rating
0,P01,T01,T01_SIM_1000-22000,SIM,3,1000-22000,3,1000-22000,test,Random,7
1,P01,T01,T01_SIM_100-1000,SIM,3,1000-22000,2,100-1000,test,Random,4
2,P01,T01,T01_SIM_1-10,SIM,0,1-10,0,1-10,test,Random,8


In [26]:
# Violin plots: for each scenario, plot by DA rank groups and by Similarity rank groups

def plot_violin(df, x_col, title):
    fig = px.violin(
        df, x=x_col, y="rating", color=x_col,
        category_orders={x_col: GROUP_LABELS},
        box=True, points="all",  # show all points
        hover_data=["pair_type", "test_id"],
        labels={x_col: "Rank Group", "rating": "Human Rating (1-9)"},
        title=title
    )
    # Make shapes fuller for discrete ratings
    fig.update_traces(
        jitter=0.25, pointpos=0.0, marker_opacity=0.28, marker_size=3,
        scalemode="width",    # constant max width
        bandwidth=0.8,         # smoother KDE on 1-9
        width=0.5,             # thicker violin body
        # spanmode="hard", span=[0.5, 9.5]  # stable support over the rating range
    )
    fig.update_layout(
        showlegend=False,
        violinmode="overlay",  # maximize per-category width
        violingap=0.0,
        width=1200, height=540,
        yaxis=dict(range=[0.5, 9.5], dtick=1, title="Human Rating (1-9)"),
        xaxis_title="Rank Group"
    )
    fig.show()

for scenario in [SCENARIO_RANDOM, SCENARIO_CORR]:
    sub = ratings[ratings["scenario"] == scenario]
    # DA rank violin
    plot_violin(sub, "da_group", f"{scenario} — Violin by DA Rank Groups")
    # Similarity rank violin
    plot_violin(sub, "sim_group", f"{scenario} — Violin by Similarity Rank Groups")

In [17]:
# Diagnostics: understand narrow shapes and the '1000-22000' DA group pattern
sub_corr = ratings[ratings["scenario"] == SCENARIO_CORR]

print("Counts per DA group (ratings, corr scenario):")
print(sub_corr.groupby("da_group").size().reindex(GROUP_LABELS))

print("\nPairs matrix DA group vs SIM group (all pairs):")
pairs_mat = (all_pairs_df
    .groupby(["da_group","sim_group"])
    .size().unstack(fill_value=0)
    .reindex(index=GROUP_LABELS, columns=GROUP_LABELS))
display(pairs_mat)

target = "1000-22000"
print(f"\nFor DA group '{target}', SIM-group composition in ratings (corr scenario):")
comp = (sub_corr[sub_corr["da_group"] == target]
        .groupby("sim_group").size()
        .reindex(GROUP_LABELS).fillna(0).astype(int))
print(comp)

print("\nMean rating by SIM group within that DA group (corr scenario):")
means = (sub_corr[sub_corr["da_group"] == target]
         .groupby("sim_group")["rating"].mean()
         .reindex(GROUP_LABELS))
print(means.round(2))

print("\nOverall mean rating by SIM group (corr scenario):")
print(sub_corr.groupby("sim_group")["rating"].mean().reindex(GROUP_LABELS).round(2))

Counts per DA group (ratings, corr scenario):
da_group
1-10            91
10-100         118
100-1000       108
1000-22000     101
22000-27900     93
27900-28000     89
dtype: int64

Pairs matrix DA group vs SIM group (all pairs):


sim_group,1-10,10-100,100-1000,1000-22000,22000-27900,27900-28000
da_group,,,,,,
1-10,24,6,3,0,3,1
10-100,12,17,11,0,1,6
100-1000,4,10,14,11,3,2
1000-22000,0,1,7,20,7,4
22000-27900,1,2,2,11,17,4
27900-28000,1,2,1,2,6,24



For DA group '1000-22000', SIM-group composition in ratings (corr scenario):
sim_group
1-10            0
10-100          3
100-1000       20
1000-22000     52
22000-27900    15
27900-28000    11
dtype: int64

Mean rating by SIM group within that DA group (corr scenario):
sim_group
1-10            NaN
10-100         6.67
100-1000       6.45
1000-22000     5.23
22000-27900    4.00
27900-28000    2.91
Name: rating, dtype: float64

Overall mean rating by SIM group (corr scenario):
sim_group
1-10           8.06
10-100         7.10
100-1000       6.23
1000-22000     5.15
22000-27900    4.03
27900-28000    3.21
Name: rating, dtype: float64


In [18]:
# Quick summaries
def summarize(df, group_col):
    return (
        df.groupby(group_col)["rating"].agg(["count", "mean", "median", "std"]).reindex(GROUP_LABELS)
    )

print("Random scenario — by DA group:")
display(summarize(ratings[ratings["scenario"] == SCENARIO_RANDOM], "da_group"))
print("\nRandom scenario — by SIM group:")
display(summarize(ratings[ratings["scenario"] == SCENARIO_RANDOM], "sim_group"))
print("\nRank-related + noise — by DA group:")
display(summarize(ratings[ratings["scenario"] == SCENARIO_CORR], "da_group"))
print("\nRank-related + noise — by SIM group:")
display(summarize(ratings[ratings["scenario"] == SCENARIO_CORR], "sim_group"))

Random scenario — by DA group:


,count,mean,median,std
da_group,,,,
1-10,91,4.527473,5.0,2.713508
10-100,118,5.008475,5.0,2.596828
100-1000,108,5.037037,5.0,2.579306
1000-22000,101,4.584158,4.0,2.478981
22000-27900,93,5.193548,5.0,2.580268
27900-28000,89,5.000000,5.0,2.713602



Random scenario — by SIM group:


,count,mean,median,std
sim_group,,,,
1-10,102,4.588235,4.5,2.652676
10-100,97,5.474227,6.0,2.479469
100-1000,96,4.875000,5.0,2.757764
1000-22000,109,4.853211,5.0,2.360264
22000-27900,89,4.921348,5.0,2.756087
27900-28000,107,4.710280,5.0,2.631483



Rank-related + noise — by DA group:


,count,mean,median,std
da_group,,,,
1-10,91,7.483516,8.0,1.642239
10-100,118,6.330508,7.0,1.970097
100-1000,108,5.953704,6.0,1.736816
1000-22000,101,5.079208,5.0,1.579134
22000-27900,93,4.709677,5.0,1.735691
27900-28000,89,3.943820,4.0,1.687922



Rank-related + noise — by SIM group:


,count,mean,median,std
sim_group,,,,
1-10,102,8.058824,8.0,0.942122
10-100,97,7.103093,7.0,1.122539
100-1000,96,6.229167,6.0,1.226713
1000-22000,109,5.146789,5.0,1.153302
22000-27900,89,4.033708,4.0,1.385390
27900-28000,107,3.214953,3.0,1.197702
